# 03 · Validez inferencial: ¿sobreviven las conclusiones clínicas?

Es la contribución central del proyecto. No basta con que los sintéticos «se parezcan»: un
investigador que analice los datos sintéticos debe llegar a las **mismas conclusiones** que con
los reales. Se mide en tres niveles: curvas de Kaplan-Meier, coeficientes de Cox y decisiones
clínicas (signo y significación).

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from synapse import datasets, generators, inference as inf, utility as ut, privacy as pv, compliance, viz
from synapse import experiment as ex
viz.style(); pd.set_option("display.precision", 3); pd.set_option("display.width", 140)
TABLES = ROOT / "results" / "tables"
metrics = pd.read_csv(TABLES / "study_metrics.csv") if (TABLES / "study_metrics.csv").exists() else None

In [2]:
d = datasets.load("gbsg2")
train, holdout = datasets.split_train_holdout(d.data, seed=0)
real_fit = inf.fit_cox(train, d)

## Curvas de supervivencia, real frente a sintético

In [3]:
syns = {g: generators.make(g, seed=0).fit(train, d.schema).sample(len(train)) for g in ("bootstrap", "marginal", "copula", "cart")}
syns["dpcopula(ε=1)"] = generators.make("dpcopula", seed=0, epsilon=1.0).fit(train, d.schema).sample(len(train))
viz.km_grid(train, syns, "GBSG2: Kaplan-Meier real frente a sintético");

In [4]:
pd.DataFrame({g: inf.compare_km(train, s).as_dict() for g, s in syns.items()}).T[
    ["km_mean_abs_diff", "rmst_real", "rmst_syn", "rmst_diff", "logrank_p"]]

,km_mean_abs_diff,rmst_real,rmst_syn,rmst_diff,logrank_p
bootstrap,0.012,1414.482,1424.062,9.581,8.713e-01
marginal,0.020,1414.482,1432.460,17.978,5.356e-01
copula,0.026,1414.482,1455.740,41.258,8.599e-01
cart,0.025,1414.482,1366.715,-47.766,4.237e-01
dpcopula(ε=1),0.226,1414.482,1864.309,449.828,7.309e-27


## Coeficientes de Cox: solapamiento de intervalos (Karr et al., 2006)

Para cada efecto se calcula cuánto se solapan los IC al 95 % real y sintético (1 = idénticos,
0 = disjuntos) y si la **decisión clínica** cambia: un efecto significativo en los datos reales que
deja de serlo, o que cambia de signo, en los sintéticos.

In [5]:
cmp = {g: inf.compare_cox(real_fit, inf.fit_cox(s, d)) for g, s in syns.items()}
pd.DataFrame({g: inf.summarize_cox(c, real_fit) for g, c in cmp.items()}).T

,ci_overlap_mean,ci_overlap_min,sign_agree_pct,sign_agree_sig_pct,signif_agree_pct,decisions_changed,abs_std_diff_mean,terms_not_estimable
bootstrap,0.703,0.416,77.778,100.000,77.778,0.0,1.265,0.0
marginal,0.418,0.000,33.333,33.333,66.667,3.0,2.572,0.0
copula,0.647,0.093,66.667,66.667,77.778,1.0,1.384,0.0
cart,0.576,0.097,55.556,100.000,88.889,0.0,1.779,0.0
dpcopula(ε=1),0.181,0.000,66.667,66.667,44.444,1.0,4.902,1.0


## Reglas de combinación: m = 5 sintéticos (Raab, Nowok y Dibben, 2016)

Igual que en la imputación múltiple, analizar varios sintéticos y combinarlos da una varianza
honesta: T = v̄·(k/n + 1/m). El intervalo se ensancha y el solapamiento con el real mejora.

In [6]:
gen = generators.make("copula", seed=0).fit(train, d.schema)
fits = [inf.fit_cox(gen.sample(len(train)), d) for _ in range(5)]
combined = inf.combine_synthetic(fits, n_real=len(train), n_syn=len(train))
c1, c5 = inf.compare_cox(real_fit, fits[0]), inf.compare_cox(real_fit, combined)
pd.DataFrame({"1 sintético": inf.summarize_cox(c1, real_fit), "m = 5 combinados": inf.summarize_cox(c5, real_fit)})

,1 sintético,m = 5 combinados
ci_overlap_mean,0.647,0.750
ci_overlap_min,0.093,0.518
sign_agree_pct,66.667,88.889
sign_agree_sig_pct,66.667,100.000
signif_agree_pct,77.778,88.889
decisions_changed,1.000,1.000
abs_std_diff_mean,1.384,1.042
terms_not_estimable,0.000,0.000


In [7]:
t = c5.join(real_fit[["lower", "upper"]].add_suffix("_real")).join(combined[["lower", "upper"]].add_suffix("_syn")).reset_index()
viz.forest(t, "GBSG2 · cópula: HR real frente a sintético (m = 5 combinados)");

## Resultados del estudio completo (10 réplicas × 4 estudios)

In [8]:
cols = ["comb_ci_overlap_mean", "ci_overlap_mean", "decisions_changed", "comb_decisions_changed", "km_mean_abs_diff", "cindex_tstr", "cindex_trtr"]
metrics.groupby(["dataset", "label"])[cols].mean().round(3)

comb_ci_overlap_mean  ci_overlap_mean  decisions_changed  comb_decisions_changed  km_mean_abs_diff  cindex_tstr  \
dataset  label                                                                                                                              
flchain  bootstrap                       0.834            0.762                0.2                     0.6             0.008        0.799   
         cart                            0.606            0.619                0.8                     0.9             0.013        0.800   
         copula                          0.572            0.512                0.6                     0.8             0.041        0.797   
         dpcopula(ε=0.5)                 0.294            0.258                1.2                     1.2             0.044        0.596   
         dpcopula(ε=1)                   0.298            0.270                1.0                     1.1             0.039        0.608   
         dpcopula(ε=10)                  0.462            0.421                0.8                     0.8             0.035        0.787   
         dpcopula(ε=5)                   0.424            0.398                0.7                     0.7             0.037        0.783   
         marginal                        0.492            0.486                2.1                     2.1             0.089        0.467   
gbsg2    bootstrap                       0.898            0.764                0.6                     0.5             0.017        0.664   
         cart                            0.808            0.669                1.5                     1.7             0.024        0.648   
         copula                          0.754            0.684                1.6                     2.3             0.031        0.645   
         dpcopula(ε=0.5)                 0.294            0.251                3.0                     3.0             0.200        0.497   
         dpcopula(ε=1)                   0.301            0.288                2.4                     2.6             0.176        0.511   
         dpcopula(ε=10)                  0.514            0.498                2.1                     2.1             0.045        0.587   
         dpcopula(ε=5)                   0.427            0.424                2.0                     2.5             0.070        0.541   
         marginal                        0.499            0.486                3.5                     3.7             0.058        0.527   
veterans bootstrap                       0.853            0.747                0.5                     0.3             0.037        0.683   
         cart                            0.764            0.697                1.0                     1.3             0.038        0.661   
         copula                          0.743            0.609                1.6                     1.9             0.042        0.651   
         dpcopula(ε=0.5)                 0.248            0.204                2.7                     2.7             0.576        0.491   
         dpcopula(ε=1)                   0.266            0.236                2.5                     2.5             0.540        0.506   
         dpcopula(ε=10)                  0.339            0.320                2.1                     2.2             0.217        0.549   
         dpcopula(ε=5)                   0.316            0.303                2.1                     2.4             0.335        0.520   
         marginal                        0.577            0.518                2.9                     3.0             0.043        0.476   
whas500  bootstrap                       0.862            0.765                0.9                     0.7             0.031        0.762   
         cart                            0.736            0.641                2.1                     2.2             0.024        0.746   
         copula                          0.693            0.626               

**Hallazgo del piloto.** Incluso el *bootstrap* (copiar pacientes reales) cambia de media
0,5 conclusiones por réplica: es la variabilidad muestral normal. Por tanto, el criterio
«0 conclusiones cambiadas» del semáforo es demasiado estricto para un único sintético y debe
calibrarse frente a ese suelo antes del preregistro.

> 📝 **Decisión para Maricel (preregistro):** fijar el umbral de `decisions_changed` y decidir si el
> semáforo usa el análisis de un sintético o el de m combinados.